## Enfoque Lambda - Speed Layer (Streaming)

> Caso real: monitoreo de transacciones de e-commerce en casi tiempo real.
> Objetivo: calcular gasto acumulado por region a medida que llegan eventos nuevos.

> Nota sobre Windows: este notebook usa una simulacion de streaming en memoria. La lectura `readStream` sobre carpetas locales requiere librerias nativas de Hadoop que no estan completas en este entorno.

### Que haremos
1. Preparar el entorno local.
2. Generar micro-lotes de eventos en memoria.
3. Levantar Spark.
4. Procesar cada micro-lote como si llegara en tiempo real.
5. Mantener un agregado acumulado por region.

In [14]:
# Celda 2: configuracion inicial
from pathlib import Path
import random
import time

BASE_DIR = Path.cwd()
random.seed(42)

print("Directorio de trabajo:", BASE_DIR)

Directorio de trabajo: c:\xampp\htdocs\codigo


In [15]:
# Celda 3: crear micro-lotes en memoria para simular eventos en streaming
regions = ["Norte", "Sur", "Centro", "Occidente"]
micro_batches = []

for _ in range(5):
    batch = []
    for _ in range(5):
        batch.append((random.choice(regions), round(random.uniform(20, 300), 2)))
    micro_batches.append(batch)

print("Micro-lotes generados:", len(micro_batches))
for i, batch in enumerate(micro_batches, start=1):
    print(f"Batch {i}: {batch}")

Micro-lotes generados: 5
Batch 1: [('Norte', 27.0), ('Centro', 88.57), ('Sur', 226.21), ('Norte', 185.34), ('Norte', 28.34)]
Batch 2: [('Sur', 85.15), ('Norte', 177.15), ('Occidente', 81.72), ('Centro', 246.64), ('Norte', 232.47)]
Batch 3: [('Sur', 215.48), ('Centro', 97.8), ('Sur', 288.02), ('Centro', 48.62), ('Occidente', 47.08)]
Batch 4: [('Centro', 189.04), ('Norte', 224.32), ('Norte', 292.47), ('Occidente', 42.06), ('Centro', 252.23)]
Batch 5: [('Centro', 181.66), ('Norte', 32.83), ('Sur', 236.46), ('Norte', 259.49), ('Norte', 126.44)]


In [20]:
# Celda 4: cargar herramientas para la simulacion incremental
import pandas as pd

print("Pandas listo para la simulacion de Speed Layer")

Pandas listo para la simulacion de Speed Layer


In [21]:
# Celda 5: definir esquema de eventos
schema = {
    "region": "string",
    "gasto_mensual": "float"
}

print(schema)
print("La Speed Layer procesara cada micro-lote con este esquema.")

{'region': 'string', 'gasto_mensual': 'float'}
La Speed Layer procesara cada micro-lote con este esquema.


In [22]:
# Celda 6: preparar la logica de agregacion incremental
eventos_acumulados = pd.DataFrame(columns=["region", "gasto_mensual"])

def actualizar_speed_layer(batch_rows):
    global eventos_acumulados

    df_batch = pd.DataFrame(batch_rows, columns=["region", "gasto_mensual"])
    eventos_acumulados = pd.concat([eventos_acumulados, df_batch], ignore_index=True)

    resultado = (
        eventos_acumulados
        .groupby("region", as_index=False)["gasto_mensual"]
        .sum()
        .rename(columns={"gasto_mensual": "total_gasto"})
        .sort_values("total_gasto", ascending=False)
    )

    return resultado

print("Logica incremental lista.")

Logica incremental lista.


In [23]:
# Celda 7: inicializar la vista de consulta
resultado_actual = pd.DataFrame(columns=["region", "total_gasto"])

print("Vista inicial preparada.")

Vista inicial preparada.


In [24]:
# Celda 8: procesar micro-lotes y observar resultados en tiempo real
for i, batch in enumerate(micro_batches, start=1):
    print(f"Iteracion {i} - batch recibido: {batch}")
    resultado_actual = actualizar_speed_layer(batch)
    print(resultado_actual.to_string(index=False))
    print()
    time.sleep(2)

Iteracion 1 - batch recibido: [('Norte', 27.0), ('Centro', 88.57), ('Sur', 226.21), ('Norte', 185.34), ('Norte', 28.34)]
region total_gasto
 Norte      240.68
   Sur      226.21
Centro       88.57

Iteracion 2 - batch recibido: [('Sur', 85.15), ('Norte', 177.15), ('Occidente', 81.72), ('Centro', 246.64), ('Norte', 232.47)]
   region total_gasto
    Norte       650.3
   Centro      335.21
      Sur      311.36
Occidente       81.72

Iteracion 3 - batch recibido: [('Sur', 215.48), ('Centro', 97.8), ('Sur', 288.02), ('Centro', 48.62), ('Occidente', 47.08)]
   region total_gasto
      Sur      814.86
    Norte       650.3
   Centro      481.63
Occidente       128.8

Iteracion 4 - batch recibido: [('Centro', 189.04), ('Norte', 224.32), ('Norte', 292.47), ('Occidente', 42.06), ('Centro', 252.23)]
   region total_gasto
    Norte     1167.09
   Centro       922.9
      Sur      814.86
Occidente      170.86

Iteracion 5 - batch recibido: [('Centro', 181.66), ('Norte', 32.83), ('Sur', 236.46), (

In [25]:
# Celda 9: fin de la simulacion
print("Simulacion finalizada.")

Simulacion finalizada.
